# installations

> Fill in a module description here

In [ ]:
# | default_exp installations

In [ ]:
# | export
import glob
import os
import platform
import subprocess
import tarfile
import zipfile
from pathlib import Path

import requests


def check_for_protoc() -> bool:
    try:
        subprocess.run(["protoc", "--version"], check=True)
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False


def install_protoc() -> None:
    if check_for_protoc():
        print("protoc is already installed.")
        return

    # Mapping of platform.system() to OS name in the download URL
    operation_systems = {"Linux": "linux", "Darwin": "osx", "Windows": "win"}

    # Mapping of platform.machine() to architecture name in the download URL
    architectures = {
        "s390x": "s390_64",
        "aarch64": "aarch_64",
        "arm64": "aarch_64",
        "ppc64le": "ppcle_64",
        "x86_64": "x86_64",
        "i386": "x86_32",
        "AMD64": "win64",  # Standard Windows 64-bit architecture
        "x86": "win32",  # Standard Windows 32-bit architecture
    }

    operation_system = operation_systems.get(platform.system(), None)
    arch = architectures.get(platform.machine(), None)

    if not operation_system or not arch:
        print("Unsupported operating system or architecture.")
        return

    if operation_system == "win":
        # For Windows, the file naming convention is `protoc-28.2-win32.zip` or `protoc-28.2-win64.zip`
        operation_system_and_arch = f"{arch}"
    else:
        # For other systems, the naming convention includes both OS and architecture
        operation_system_and_arch = f"{arch}-{operation_system}"

    url = f"https://github.com/protocolbuffers/protobuf/releases/download/v28.2/protoc-28.2-{operation_system_and_arch}.zip"

    zip_filename = Path("protoc.zip").absolute()
    if not download_archive(url, zip_filename):
        print(f"Failed to download protoc from {url}.")
        return
    extract_dir = Path("protoc").absolute()
    with zipfile.ZipFile(zip_filename, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extracted protoc files to {extract_dir}.")

    zip_filename.unlink()
    print(f"Removed the zip file: {zip_filename}")
    protoc_bin_path = extract_dir.joinpath("bin")
    protoc_path = protoc_bin_path.joinpath("protoc")
    protoc_path.chmod(0o755)
    os.environ["PATH"] = os.pathsep.join([str(protoc_bin_path), os.environ["PATH"]])


def download_archive(url: str, dest_file_path: Path) -> bool:
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(dest_file_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
        return True
    else:
        return False


def check_for_rust() -> bool:
    try:
        subprocess.run(["rustc", "--version"], check=True)
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False


def install_rust():
    if check_for_rust():
        print("Rust is already installed.")
        return

    operation_systems = {
        "Linux": "unknown-linux-gnu",
        "Darwin": "apple-darwin",
        "Windows": "pc-windows-gnu",
    }

    architectures = {
        "s390x": "s390x",
        "aarch64": "aarch64",
        "arm64": "aarch64",
        "x86_64": "x86_64",
        "AMD64": "x86_64",
    }

    operation_system = operation_systems.get(platform.system(), None)
    arch = architectures.get(platform.machine(), None)

    if not operation_system or not arch:
        print("Unsupported operating system or architecture.")
        return
    operation_system_and_arch = f"{arch}-{operation_system}"
    url = f"https://static.rust-lang.org/dist/rust-1.81.0-{operation_system_and_arch}.tar.xz"

    tar_xz_filename = Path("rust.tar.xz").absolute()
    if not download_archive(url, tar_xz_filename):
        print(f"Failed to download Rust from {url}.")
        return

    extract_dir = Path("rust").absolute()
    with tarfile.open(tar_xz_filename, "r:xz") as tar:
        tar.extractall(extract_dir)
    print(f"Extracted Rust files to {extract_dir}.")
    tar_xz_filename.unlink()
    print(f"Removed the tar.xz file: {tar_xz_filename}")
    dirs = glob.glob(f"{extract_dir}/*/*/bin")
    os.environ["PATH"] = os.pathsep.join(dirs + [os.environ["PATH"]])

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()